## **Random Forest Regressor** ##

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [22]:
# 1. Load and sort data
df = pd.read_csv("data.csv")
df["From"] = pd.to_datetime(df["From"])
df = df.sort_values("From").reset_index(drop=True)

mete_col = ["Diurnal temperature range (°C)", "Mean temperature (°C)", "Mean humidity (%)", "Solar radiation (MJ/m²)", "Air pressure (hPa)", "Dew point temperature (°C)", "Wind speed (km/h)", "Mean UV"]
# flu_col = ["H1", "H3", "B", "AandB"]
flu_col = ["H1", "H3", "AandB"]


for col in flu_col: 
    tmp_df = df.copy()

    if col == "H1":
        for n in range(4):
            tmp_df[f"{col}_Lag{n+1}"] = tmp_df[col].shift(n+1)
        tmp_df[f"{col}_volatility"] = tmp_df[col].shift(1).rolling(window=4).std()
        tmp_df[f"{col}_rolling_mean"] = tmp_df[col].shift(1).rolling(window=4).mean()
    else:
        for n in range(2):
            tmp_df[f"{col}_Lag{n+1}"] = tmp_df[col].shift(n+1)
        tmp_df[f"{col}_volatility"] = tmp_df[col].shift(1).rolling(window=4).std()
        tmp_df[f"{col}_rolling_mean"] = tmp_df[col].shift(1).rolling(window=4).mean()

    tmp_df = tmp_df.dropna().reset_index(drop=True)

    X = tmp_df.drop(columns=["From"]+flu_col)


    print(f"{'='*70} {col} {'='*70}")
    y = tmp_df[col]

    train_end = int(tmp_df.shape[0] * 0.8)
    X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
    y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]

    y_train_log = np.log1p(y_train)

    grid = {
        "n_estimators": [x for x in range(100, 1000, 100)],
        "max_features": ["sqrt", "log2", 0.8],
        "max_depth": [3, 4, 5, 6, 7, 8, 9, 10],
        "random_state": [18, 24, 42],
        "min_samples_split": [2, 5], 
        "min_samples_leaf": [5, 10, 20, 30]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    rfr = RandomForestRegressor(random_state=42)
    rfr_cv = RandomizedSearchCV(estimator=rfr, param_distributions=grid, cv=tscv, scoring="r2", n_jobs=-1)

    rfr_cv.fit(X_train, y_train_log)
    print(f"Best Parameters: {rfr_cv.best_params_}")

    print(f"\nFeature importances: ")
    model = rfr_cv.best_estimator_
    importance = pd.Series(model.feature_importances_, index=X.columns)
    print(importance.sort_values(ascending=False))

    model.fit(X_train, y_train_log)

    y_pred_log = model.predict(X_train)
    y_pred = np.expm1(y_pred_log)
    print(f"\nModel metrics with train data: ")
    print(f"MAE: {mean_absolute_error(y_train, y_pred):.4f}")
    print(f"MSE: {mean_squared_error(y_train, y_pred):.4f}")
    print(f"R2:  {r2_score(y_train, y_pred):.4f}")

    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    print(f"\nModel metrics with test data: ")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
    print(f"R2:  {r2_score(y_test, y_pred):.4f}")

====================================================================== H1 ======================================================================
Best Parameters: {'random_state': 24, 'n_estimators': 900, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 0.8, 'max_depth': 4}

Feature importances: 
H1_Lag1                           0.730240
H1_rolling_mean                   0.159894
H1_Lag2                           0.088370
H1_volatility                     0.008355
H1_Lag3                           0.003519
Week                              0.002362
H1_Lag4                           0.001287
B                                 0.001133
Mean temperature (°C)             0.000810
Dew point temperature (°C)        0.000782
Wind speed (km/h)                 0.000718
Diurnal temperature range (°C)    0.000607
Solar radiation (MJ/m²)           0.000553
Air pressure (hPa)                0.000524
Mean humidity (%)                 0.000275
Population                        0.000199
L